# Intelligent Customer Segmentation & Behavioral Analytics
## Notebook 02: Feature Engineering & Preprocessing Pipeline

### Overview
Unsupervised clustering models are sensitive to feature scales and feature engineering. This notebook transforms raw transactional metrics into rich behavioral RFM (Recency, Frequency, Monetary) indicators, builds composite engagement scores, and scales features.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Import custom modular preprocessing pipeline
sys.path.append(os.path.join('..', 'training'))
from preprocessing import engineer_features, CLUSTERING_FEATURES, fit_scaler

### 1. Load Raw Customer Data

In [ ]:
raw_path = os.path.join('..', 'data', 'customer_data.csv')
df = pd.read_csv(raw_path)
print(f"Loaded raw dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

### 2. Derive Behavioral Features
We compute:
- **Recency**: Days since last inquiry or booking.
- **Frequency**: Monthly booking velocity (`Total_Bookings / Tenure_Months`).
- **Monetary**: `Total_Spend_INR` and `Avg_Booking_Value_INR`.
- **Engagement Score**: Weighted composite of active touchpoints (WhatsApp 35%, App 30%, Web 20%, Wishlist 15%).
- **Discount Dependency**: Ratio of promo code usage.
- **Shortlist Conversion Ratio**: Conversion rate from shortlisted venues to confirmed bookings.

In [ ]:
featured_df = engineer_features(df)
print("Newly engineered feature preview:")
featured_df[['Customer_ID', 'Recency_Days', 'Frequency_Per_Month', 'Engagement_Score', 'Discount_Dependency']].head(5)

### 3. Distribution of Derived Behavioral Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(featured_df['Frequency_Per_Month'], kde=True, color='#2A9D8F', ax=axes[0, 0])
axes[0, 0].set_title('Frequency: Bookings per Month')

sns.histplot(featured_df['Engagement_Score'], kde=True, color='#E76F51', ax=axes[0, 1])
axes[0, 1].set_title('Composite Engagement Score')

sns.scatterplot(x='Engagement_Score', y='Total_Spend_INR', data=featured_df, alpha=0.5, color='#1D3557', ax=axes[1, 0])
axes[1, 0].set_title('Engagement Score vs Total Spend')

sns.scatterplot(x='Discount_Dependency', y='Abandonment_Rate', data=featured_df, alpha=0.5, color='#E63946', ax=axes[1, 1])
axes[1, 1].set_title('Discount Dependency vs Abandonment Rate')

plt.tight_layout()
plt.show()

### 4. Feature Selection for Unsupervised Clustering
We select the core feature subset that encapsulates distinct dimensions (Recency, Frequency, Monetary, Engagement, Sensitivity) without excessive collinearity.

In [ ]:
print(f"Selected {len(CLUSTERING_FEATURES)} Core Clustering Features:")
for f in CLUSTERING_FEATURES:
    print(f" - {f}")

X = featured_df[CLUSTERING_FEATURES]
X.describe().T

### 5. Feature Scaling (StandardScaler)
Distance-based algorithms (K-Means, DBSCAN, Agglomerative) require zero mean and unit variance.

In [ ]:
scaler = fit_scaler(X)
X_scaled = scaler.transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=CLUSTERING_FEATURES)
print("Scaled feature distributions (Mean ~ 0, Std ~ 1):")
scaled_df.describe().round(3).T[['mean', 'std', 'min', 'max']]

### Summary
- Features engineered and validated.
- Numerical scales normalized.
- The transformed matrix `X_scaled` is ready for clustering algorithms in `03_Clustering.ipynb`.